# AADS-ULoRA v5.5 - Complete Auto-Training Pipeline

**One-click end-to-end training from setup to results**

This notebook automatically:
1. ✅ Sets up Colab environment (GPU, dependencies, Drive)
2. ✅ Clones/syncs the repository
3. ✅ Prepares training data
4. ✅ Runs Phase 1 Training (DoRA)
5. ✅ Runs Phase 2 Training (SD-LoRA)
6. ✅ Runs Phase 3 Training (CoNeC-LoRA)
7. ✅ Validates all models
8. ✅ Generates performance reports

**No manual intervention needed** - just run all cells in order.

---
**Expected Runtime:** 8-12 hours (depending on GPU)  
**Required:** GPU with 16GB+ VRAM  
**Storage:** 100GB free space on Google Drive

⏱️ **Estimated breakdown:**
- Setup: 10 min
- Data Prep: 20 min
- Phase 1 (DoRA): 3-4 hours
- Phase 2 (SD-LoRA): 2-3 hours
- Phase 3 (CoNeC): 2-3 hours
- Validation & Reports: 30 min

---

## 1️⃣ INITIAL SETUP & ENVIRONMENT

## ⚙️ CONFIGURATION: Enter Your Parameters

**Before training begins, configure these settings:**


In [ ]:
# Install ipywidgets for interactive UI
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'], check=True)

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Create configuration widgets
print("📝 AADS-ULoRA Training Configuration")
print("="*60)

# Crop selection
crop_label = widgets.Label(value="🌾 Select Crop(s) for Training:")
crop_options = ['tomato', 'potato', 'wheat', 'custom']
crop_multi = widgets.SelectMultiple(
    options=crop_options,
    value=['tomato'],
    description='Crops:',
    disabled=False
)

# Custom crop name (if needed)
custom_crop_input = widgets.Text(
    value='',
    placeholder='Enter custom crop name',
    description='Custom:',
    disabled=False,
    style={'description_width': '80px'}
)

# Google Drive dataset path
dataset_path_label = widgets.Label(value="📁 Google Drive Dataset (Class-Root) Configuration:")
dataset_path_input = widgets.Text(
    value='/content/drive/MyDrive/your_dataset_root',
    placeholder='/content/drive/MyDrive/your_dataset_root',
    description='Class-Root Path:',
    disabled=False,
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

# Training parameters
params_label = widgets.Label(value="⚙️ Training Parameters:")

batch_size_slider = widgets.IntSlider(
    value=32,
    min=8,
    max=128,
    step=8,
    description='Batch Size:',
    style={'description_width': '120px'}
)

learning_rate_slider = widgets.FloatSlider(
    value=1e-4,
    min=1e-6,
    max=1e-2,
    step=1e-5,
    description='Learning Rate:',
    style={'description_width': '120px'}
)

num_epochs_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    step=1,
    description='Epochs per Phase:',
    style={'description_width': '120px'}
)

# Phase selection
phases_label = widgets.Label(value="🎯 Training Phases:")
phase1_checkbox = widgets.Checkbox(value=True, description='Phase 1: DoRA', indent=False)
phase2_checkbox = widgets.Checkbox(value=True, description='Phase 2: SD-LoRA', indent=False)
phase3_checkbox = widgets.Checkbox(value=True, description='Phase 3: CoNeC-LoRA', indent=False)

# Resume from checkpoint
resume_label = widgets.Label(value="💾 Checkpoint & Resume:")
resume_checkbox = widgets.Checkbox(value=True, description='Resume from checkpoint if available', indent=False)
checkpoint_path_input = widgets.Text(
    value='',
    placeholder='Leave empty for auto-detection',
    description='Checkpoint:',
    disabled=False,
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

# Output directory
output_label = widgets.Label(value="📊 Output Configuration:")
output_dir_input = widgets.Text(
    value='/content/drive/MyDrive/aads_ulora_results',
    placeholder='/content/drive/MyDrive/aads_ulora_results',
    description='Output Dir:',
    disabled=False,
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

# Advanced options
advanced_label = widgets.Label(value="🔧 Advanced Options:")
device_dropdown = widgets.Dropdown(
    options=['cuda', 'cpu'],
    value='cuda',
    description='Device:',
    disabled=False,
    style={'description_width': '120px'}
)

mixed_precision_checkbox = widgets.Checkbox(value=True, description='Mixed Precision (FP16)', indent=False)
validate_checkbox = widgets.Checkbox(value=True, description='Run validation after training', indent=False)
save_best_only_checkbox = widgets.Checkbox(value=True, description='Save best model only', indent=False)

# Display configuration UI
display(crop_label)
display(crop_multi)
display(custom_crop_input)
display(HTML("<hr>"))

display(dataset_path_label)
display(dataset_path_input)
display(HTML("<hr>"))

display(params_label)
display(batch_size_slider)
display(learning_rate_slider)
display(num_epochs_slider)
display(HTML("<hr>"))

display(phases_label)
display(phase1_checkbox)
display(phase2_checkbox)
display(phase3_checkbox)
display(HTML("<hr>"))

display(resume_label)
display(resume_checkbox)
display(checkpoint_path_input)
display(HTML("<hr>"))

display(output_label)
display(output_dir_input)
display(HTML("<hr>"))

display(advanced_label)
display(device_dropdown)
display(mixed_precision_checkbox)
display(validate_checkbox)
display(save_best_only_checkbox)
display(HTML("<hr>"))

# Display current configuration
def display_config():
    print("\n✅ Configuration Summary:")
    print(f"  Crops: {', '.join(crop_multi.value)}")
    if custom_crop_input.value:
        print(f"  Custom Crop: {custom_crop_input.value}")
    print(f"  Dataset Path: {dataset_path_input.value}")
    print(f"  Batch Size: {batch_size_slider.value}")
    print(f"  Learning Rate: {learning_rate_slider.value}")
    print(f"  Epochs per Phase: {num_epochs_slider.value}")
    print(f"  Phases: Phase 1={phase1_checkbox.value}, Phase 2={phase2_checkbox.value}, Phase 3={phase3_checkbox.value}")
    print(f"  Resume from Checkpoint: {resume_checkbox.value}")
    print(f"  Output Directory: {output_dir_input.value}")
    print(f"  Device: {device_dropdown.value}")
    print(f"  Mixed Precision: {mixed_precision_checkbox.value}")
    print(f"  Validation: {validate_checkbox.value}")
    print(f"  Save Best Only: {save_best_only_checkbox.value}")
    print("\nReady to start training! Continue to the next cell.")

display_config()


In [ ]:
import os
import sys
import subprocess
import json
from pathlib import Path
from datetime import datetime
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

selected_crops = list(crop_multi.value)
if not selected_crops:
    raise ValueError('Select at least one crop before starting training.')
if 'custom' in selected_crops and not custom_crop_input.value.strip():
    raise ValueError("'custom' crop selected but custom crop name is empty.")

if checkpoint_path_input.value.strip():
    CHECKPOINT_BASE = Path(checkpoint_path_input.value.strip()).expanduser()
else:
    CHECKPOINT_BASE = Path(output_dir_input.value) / '.checkpoints'
CHECKPOINT_BASE.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path(output_dir_input.value).expanduser()
MODELS_DIR = OUTPUT_DIR / 'models'
LOGS_DIR = OUTPUT_DIR / 'logs'
for d in (OUTPUT_DIR, MODELS_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

PIPELINE_START_TIME = datetime.now()
PIPELINE_STATUS = {'stages': {}, 'errors': [], 'warnings': []}

TRAINING_CONFIG = {
    'crops': selected_crops,
    'custom_crop': custom_crop_input.value.strip(),
    'dataset_path': dataset_path_input.value.strip(),
    'source_dataset_path': dataset_path_input.value.strip(),
    'prepared_dataset_path': None,
    'batch_size': batch_size_slider.value,
    'learning_rate': learning_rate_slider.value,
    'num_epochs': num_epochs_slider.value,
    'phases': {'phase1': phase1_checkbox.value, 'phase2': phase2_checkbox.value, 'phase3': phase3_checkbox.value},
    'resume_from_checkpoint': resume_checkbox.value,
    'checkpoint_path': checkpoint_path_input.value.strip(),
    'output_directory': str(OUTPUT_DIR),
    'checkpoint_directory': str(CHECKPOINT_BASE),
    'device': device_dropdown.value,
    'mixed_precision': mixed_precision_checkbox.value,
    'validate': validate_checkbox.value,
    'save_best_only': save_best_only_checkbox.value,
}

TRAINING_CONFIG_PATH = OUTPUT_DIR / 'training_config.json'
with open(TRAINING_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump({'created_at': PIPELINE_START_TIME.isoformat(), 'config': TRAINING_CONFIG}, f, indent=2)

print('AADS-ULoRA Complete Auto-Training Pipeline')
print(f"Started at {PIPELINE_START_TIME.strftime('%Y-%m-%d %H:%M:%S')}")
print('=' * 60)
print()
print('Using Configuration:')
for key, value in TRAINING_CONFIG.items():
    if key != 'phases':
        print(f'  {key}: {value}')
    else:
        print('  phases:')
        for phase, enabled in value.items():
            print(f'    - {phase}: {enabled}')
print()
print(f'Checkpoint Directory: {CHECKPOINT_BASE}')
print(f'Models Directory: {MODELS_DIR}')
print(f'Logs Directory: {LOGS_DIR}')
print(f'Config Record: {TRAINING_CONFIG_PATH}')


In [ ]:
import json
from pathlib import Path

# Checkpoint management system
class CheckpointManager:
    """Manages training checkpoints for easy recovery and resumption."""
    
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_log = self.checkpoint_dir / 'checkpoint_log.json'
        self.load_checkpoint_log()
    
    def load_checkpoint_log(self):
        """Load existing checkpoint log or create new one."""
        if self.checkpoint_log.exists():
            with open(self.checkpoint_log, 'r', encoding='utf-8') as f:
                self.checkpoints = json.load(f)
        else:
            self.checkpoints = {
                'setup': None,
                'data_prep': None,
                'phase1': None,
                'phase2': None,
                'phase3': None,
                'validation': None,
                'monitoring': None,
            }
            self.save_checkpoint_log()
    
    def save_checkpoint_log(self):
        """Save checkpoint log to file."""
        with open(self.checkpoint_log, 'w', encoding='utf-8') as f:
            json.dump(self.checkpoints, f, indent=2)
    
    def save_checkpoint(self, stage_name, details=None):
        """Save checkpoint for a training stage."""
        timestamp = datetime.now().isoformat()
        self.checkpoints[stage_name] = {
            'timestamp': timestamp,
            'completed': True,
            'details': details or {}
        }
        self.save_checkpoint_log()
        return timestamp
    
    def get_checkpoint(self, stage_name):
        """Get checkpoint info for a stage."""
        return self.checkpoints.get(stage_name)
    
    def has_checkpoint(self, stage_name):
        """Check if checkpoint exists for a stage."""
        cp = self.get_checkpoint(stage_name)
        return cp is not None and cp.get('completed', False)
    
    def display_checkpoint_status(self):
        """Display current checkpoint status."""
        print("\n📊 CHECKPOINT STATUS:")
        print("="*60)
        for stage, cp in self.checkpoints.items():
            if cp and cp.get('completed'):
                timestamp = cp.get('timestamp', 'unknown')
                status = "✅ COMPLETED"
                print(f"  {stage:12} {status:20} ({timestamp})")
            else:
                print(f"  {stage:12} {'⊘ PENDING':20}")
    
    def clear_checkpoints(self, stages=None):
        """Clear specific checkpoints or all."""
        if stages is None:
            stages = list(self.checkpoints.keys())
        for stage in stages:
            if stage in self.checkpoints:
                self.checkpoints[stage] = None
        self.save_checkpoint_log()
        print(f"✓ Cleared checkpoints for: {', '.join(stages)}")

# Initialize checkpoint manager
checkpoint_manager = CheckpointManager(TRAINING_CONFIG['checkpoint_directory'])

print("\n✅ Checkpoint system initialized")
checkpoint_manager.display_checkpoint_status()

# Save setup checkpoint
checkpoint_manager.save_checkpoint('setup', {
    'configuration': TRAINING_CONFIG,
    'time': datetime.now().isoformat()
})
print("\n✅ Setup checkpoint saved")


## 💾 Checkpoint System for Reliable Training

The pipeline includes automatic **checkpointing at every major step** to enable:
- ✅ **Recovery from failures** - If training interrupts, resume from the last successful checkpoint
- ✅ **Save compute time** - Completed phases are automatically skipped on re-runs
- ✅ **Easy troubleshooting** - Clear indication of which phases completed vs. failed
- ✅ **Reproducibility** - Configuration and progress logged for each checkpoint

### Checkpoint Stages:
1. **Setup** - Initial environment configuration
2. **Data Prep** - Dataset preparation and verification
3. **Phase 1** - DoRA adapter training
4. **Phase 2** - SD-LoRA adapter training
5. **Phase 3** - CoNeC-LoRA adapter training
6. **Validation** - Model validation and testing
7. **Monitoring** - Performance reports generation

### How to Use Checkpoints:
- **First run**: All phases execute → all checkpoints created
- **Resume run**: Previously completed phases skip automatically
- **Partial re-training**: Modify configuration → uncompleted phases re-run
- **Clear checkpoints**: Use checkpoint manager to reset specific or all stages

### Checkpoint Location:
All checkpoints stored in: `.checkpoints/checkpoint_log.json` (relative to output directory)


In [ ]:
# Checkpoint Management Helpers (will be available after setup cell runs)

def print_checkpoint_help():
    """Print help information about using checkpoints."""
    print("""
    CHECKPOINT MANAGEMENT COMMANDS
    ==============================
    
    After the setup cell runs, you can use these commands:
    
    # Check status of all checkpoints
    checkpoint_manager.display_checkpoint_status()
    
    # Clear a specific checkpoint
    checkpoint_manager.clear_checkpoints(['phase1'])  # Clear only phase1
    
    # Clear multiple checkpoints
    checkpoint_manager.clear_checkpoints(['phase1', 'phase2', 'validation'])
    
    # Clear all checkpoints and start over
    checkpoint_manager.clear_checkpoints()  # Clears everything
    
    # Get checkpoint information
    cp = checkpoint_manager.get_checkpoint('phase1')
    print(cp)  # Shows timestamp and details
    
    # Check if a specific stage is completed
    if checkpoint_manager.has_checkpoint('phase1'):
        print("Phase 1 is done!")
    else:
        print("Phase 1 needs to be run")
    """)

print_checkpoint_help()


### Step 1.1: Detect GPU and System Info

In [ ]:
try:
    import torch

    requested_device = TRAINING_CONFIG.get('device', 'cuda')
    cuda_available = torch.cuda.is_available()

    if requested_device == 'cuda' and not cuda_available:
        print("CUDA requested but no GPU detected. Falling back to CPU.")
        TRAINING_CONFIG['device'] = 'cpu'
    elif requested_device == 'cpu':
        print("CPU mode selected by configuration.")
        if cuda_available:
            print("   GPU is available, but CPU was explicitly requested.")
    else:
        TRAINING_CONFIG['device'] = 'cuda'

    if torch.cuda.is_available():
        device_count = torch.cuda.device_count()
        current_device = torch.cuda.current_device()
        device_name = torch.cuda.get_device_name(current_device)

        try:
            smi_output = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                encoding='utf-8'
            ).strip().splitlines()[0]
            memory_gb = int(smi_output) / 1024
        except Exception:
            memory_gb = torch.cuda.get_device_properties(current_device).total_memory / (1024**3)

        print("GPU DETECTED")
        print(f"   Device: {device_name}")
        print(f"   Memory: {memory_gb:.1f} GB")
        print(f"   PyTorch: {torch.__version__}")
        print(f"   CUDA: {torch.version.cuda}")
        print(f"   Device Count: {device_count}")
    else:
        print("Running without CUDA GPU")
        print(f"   PyTorch: {torch.__version__}")

    print(f"Effective device: {TRAINING_CONFIG['device']}")

except Exception as e:
    print(f"Error detecting runtime device: {e}")
    raise


### Step 1.2: Mount Google Drive

In [ ]:
import shutil

IS_COLAB = 'google.colab' in sys.modules
print("Runtime environment detection...")

if IS_COLAB:
    from google.colab import drive

    print("Mounting Google Drive...")
    try:
        MOUNT_POINT = Path('/content/drive')
        if (MOUNT_POINT / 'MyDrive').exists():
            print("Google Drive already mounted")
        else:
            MOUNT_POINT.mkdir(parents=True, exist_ok=True)
            if any(MOUNT_POINT.iterdir()):
                print("/content/drive contains stale files, clearing before mount")
                for entry in list(MOUNT_POINT.iterdir()):
                    if entry.is_dir():
                        shutil.rmtree(entry, ignore_errors=True)
                    else:
                        entry.unlink(missing_ok=True)
            drive.mount(str(MOUNT_POINT), force_remount=False)

        DRIVE_PATH = MOUNT_POINT / 'MyDrive'
        if not DRIVE_PATH.exists():
            raise RuntimeError("Drive mount succeeded but /content/drive/MyDrive is missing.")

        PROJECT_ROOT = DRIVE_PATH / 'aads_ulora'
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully")
        print(f"Project directory: {PROJECT_ROOT}")
    except Exception as e:
        print(f"Failed to mount Drive: {e}")
        raise
else:
    PROJECT_ROOT = Path.cwd().resolve()
    print("Non-Colab environment detected (local VSCode/Jupyter)")
    print(f"Using local project root: {PROJECT_ROOT}")

WORKSPACE_ROOT = Path('/content/aads_ulora') if IS_COLAB else (PROJECT_ROOT / '.runtime_tmp' / 'aads_ulora')
for rel_dir in ('config', 'data', 'models', 'checkpoints', 'logs', 'outputs'):
    (WORKSPACE_ROOT / rel_dir).mkdir(parents=True, exist_ok=True)
print(f"Workspace root: {WORKSPACE_ROOT}")


### Step 1.3: Clone/Update Repository

In [ ]:
import shutil

print("Repository setup...")

def _is_repo_root(path: Path) -> bool:
    return (path / '.git').exists() and (path / 'colab_notebooks').exists() and (path / 'src').exists()

try:
    if IS_COLAB:
        REPO_PATH = PROJECT_ROOT / 'bitirmeprojesi'
        print(f"Target repository path: {REPO_PATH}")

        if REPO_PATH.exists() and (REPO_PATH / '.git').exists():
            print("Repository already cloned. Updating...")
            os.chdir(REPO_PATH)
            subprocess.run(['git', 'pull', 'origin', 'master'], check=True, capture_output=True)
            print("Repository updated")
        elif REPO_PATH.exists() and not (REPO_PATH / '.git').exists():
            raise RuntimeError(f"Existing path is not a git repository: {REPO_PATH}")
        else:
            print("Cloning repository...")
            subprocess.run(
                ['git', 'clone', 'https://github.com/EfeErim/bitirmeprojesi.git', str(REPO_PATH)],
                check=True,
                capture_output=True
            )
            os.chdir(REPO_PATH)
            print("Repository cloned")
    else:
        candidates = [PROJECT_ROOT, PROJECT_ROOT / 'bitirmeprojesi', Path.cwd().resolve(), Path.cwd().resolve().parent]
        REPO_PATH = None
        for candidate in candidates:
            if _is_repo_root(candidate):
                REPO_PATH = candidate
                break

        if REPO_PATH is None:
            raise RuntimeError(
                "Could not locate repository root in local mode. "
                "Open this notebook from repo root or set PROJECT_ROOT accordingly."
            )

        os.chdir(REPO_PATH)
        print(f"Local repository detected: {REPO_PATH}")

except Exception as e:
    print(f"Repository setup failed: {e}")
    raise

print(f"Working directory: {os.getcwd()}")


### Step 1.4: Install Dependencies

In [ ]:
print("📦 Installing dependencies...")

def run_pip(cmd, step_name):
    """Run pip with captured logs so Colab surfaces the actual failing package."""
    print(f"\n[{step_name}] {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ {step_name} failed (exit={result.returncode})")
        if result.stdout:
            print("\n--- pip stdout (tail) ---")
            print(result.stdout[-4000:])
        if result.stderr:
            print("\n--- pip stderr (tail) ---")
            print(result.stderr[-4000:])
        raise subprocess.CalledProcessError(result.returncode, cmd)
    return result

# Upgrade pip first
run_pip([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], 'pip-upgrade')

# Prefer canonical requirements at repo root.
canonical_req = REPO_PATH / 'requirements_colab.txt'
mirror_req = REPO_PATH / 'colab_notebooks' / 'requirements_colab.txt'
default_req = REPO_PATH / 'requirements.txt'

if canonical_req.exists():
    req_file = canonical_req
elif mirror_req.exists():
    req_file = mirror_req
else:
    req_file = default_req

print(f"Installing from {req_file}...")
try:
    run_pip([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)], 'install-requirements')
except subprocess.CalledProcessError:
    # Retry once with optional/dev packages removed to keep training path usable.
    optional_pkgs = {
        'groundingdino-hf',
        'google-colab',
        'jupyter',
        'pytest',
        'pytest-cov',
        'black',
        'flake8',
        'mypy',
    }
    fallback_req = REPO_PATH / '.runtime_requirements_colab.txt'
    filtered_lines = []
    for raw_line in req_file.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or line.startswith('-r '):
            filtered_lines.append(raw_line)
            continue
        pkg = line.split(';')[0]
        pkg = pkg.split('==')[0].split('>=')[0].split('<=')[0].split('~=')[0]
        pkg = pkg.split('>')[0].split('<')[0].strip().lower()
        if pkg in optional_pkgs:
            print(f"Skipping optional package on fallback: {pkg}")
            continue
        filtered_lines.append(raw_line)
    fallback_req.write_text('\n'.join(filtered_lines) + '\n', encoding='utf-8')
    print(f"Retrying install with fallback requirements: {fallback_req}")
    run_pip([sys.executable, '-m', 'pip', 'install', '-r', str(fallback_req)], 'install-requirements-fallback')

print("✅ Dependencies installed")

### Step 1.5: Setup Python Path and Imports

In [ ]:
# Add repo to Python path
sys.path.insert(0, str(REPO_PATH))

# Test imports
try:
    from src.core.config_manager import config_manager, get_config
    from src.dataset.colab_datasets import ColabCropDataset, ColabDomainShiftDataset
    from src.training.colab_phase1_training import ColabPhase1Trainer
    from src.training.colab_phase2_sd_lora import ColabPhase2Trainer
    from src.training.colab_phase3_conec_lora import ColabPhase3Trainer
    import torch
    import torchvision
    from PIL import Image
    
    print("✅ All imports successful")
    print(f"   - PyTorch: {torch.__version__}")
    print(f"   - TorchVision: {torchvision.__version__}")
except Exception as e:
    print(f"❌ Import error: {e}")
    raise

### Step 1.6: Initialize Configuration

In [ ]:
print()
print('DATA PREPARATION PHASE')
print('=' * 60)

data_start_time = datetime.now()

source_dataset_root = Path(TRAINING_CONFIG.get('source_dataset_path') or TRAINING_CONFIG['dataset_path']).expanduser()
workspace_root = WORKSPACE_ROOT
prepared_dataset_root = workspace_root / 'data' / 'dataset_splits'
metadata_path = workspace_root / 'data' / 'dataset_metadata.json'
workspace_config_dir = workspace_root / 'config'

checkpoint_available = TRAINING_CONFIG.get('resume_from_checkpoint', True) and checkpoint_manager.has_checkpoint('data_prep')
cp = checkpoint_manager.get_checkpoint('data_prep') if checkpoint_available else None

image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
split_names = {'train', 'val', 'test'}

def _has_images(split_path: Path) -> bool:
    return any(p.is_file() and p.suffix.lower() in image_exts for p in split_path.rglob('*'))

def _norm(path_like):
    try:
        return str(Path(path_like).expanduser().resolve())
    except Exception:
        return str(Path(path_like).expanduser())

prepared_splits_available = all((prepared_dataset_root / split).exists() and _has_images(prepared_dataset_root / split) for split in ('train', 'val', 'test'))

metadata_matches_source = False
if metadata_path.exists():
    try:
        with open(metadata_path, 'r', encoding='utf-8') as f:
            existing_metadata = json.load(f)
        metadata_source = existing_metadata.get('source_dataset_path')
        metadata_layout = existing_metadata.get('source_layout') == 'class_root'
        metadata_matches_source = bool(metadata_source) and _norm(metadata_source) == _norm(source_dataset_root) and metadata_layout
    except Exception:
        metadata_matches_source = False

checkpoint_matches_source = False
if cp:
    cp_details = cp.get('details', {}) if isinstance(cp, dict) else {}
    cp_source = cp_details.get('source_dataset_path')
    cp_prepared = cp_details.get('prepared_dataset_path')
    checkpoint_matches_source = bool(cp_source) and _norm(cp_source) == _norm(source_dataset_root)
    if cp_prepared:
        checkpoint_matches_source = checkpoint_matches_source and (_norm(cp_prepared) == _norm(prepared_dataset_root))

if checkpoint_available and prepared_splits_available and metadata_matches_source and checkpoint_matches_source:
    print(f"Data prep already completed at {cp['timestamp']}")
    print('Skipping data preparation (valid checkpoint + matching non-empty prepared splits found)')
else:
    reasons = []
    if not checkpoint_available:
        reasons.append('no data_prep checkpoint (or resume disabled)')
    if not prepared_splits_available:
        reasons.append('prepared train/val/test splits missing or empty')
    if not metadata_matches_source:
        reasons.append('dataset metadata missing/mismatched source path')
    if checkpoint_available and not checkpoint_matches_source:
        reasons.append('checkpoint source path does not match current source dataset')
    if reasons:
        print('Rebuilding splits because: ' + '; '.join(reasons))

    import random
    import shutil

    for rel_dir in ('config', 'data', 'models', 'checkpoints', 'logs', 'outputs'):
        (workspace_root / rel_dir).mkdir(parents=True, exist_ok=True)

    workspace_src = workspace_root / 'src'
    if not workspace_src.exists():
        try:
            os.symlink(REPO_PATH / 'src', workspace_src)
        except Exception:
            shutil.copytree(REPO_PATH / 'src', workspace_src, dirs_exist_ok=True)

    print(f"Source dataset path: {source_dataset_root}")
    print('Required source layout: <root>/<class_name>/<images>')
    print('Split policy: 80/10/10 (train/val/test)')

    if not source_dataset_root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {source_dataset_root}")
    if not source_dataset_root.is_dir():
        raise NotADirectoryError(f"Dataset path is not a directory: {source_dataset_root}")

    found_split_dirs = [d.name for d in source_dataset_root.iterdir() if d.is_dir() and d.name.lower() in split_names]
    if found_split_dirs:
        raise RuntimeError('Dataset path must point to class-root layout only (no train/val/test at root). ' + f'Found split folders: {found_split_dirs}')

    class_dirs = sorted([d for d in source_dataset_root.iterdir() if d.is_dir()], key=lambda p: p.name.lower())
    if not class_dirs:
        raise RuntimeError(f"No class folders found under dataset root: {source_dataset_root}")

    class_to_images = {}
    for class_dir in class_dirs:
        files = [p for p in class_dir.rglob('*') if p.is_file() and p.suffix.lower() in image_exts]
        if not files:
            raise RuntimeError(f"Class folder has no images: {class_dir}")
        class_to_images[class_dir.name] = files

    if prepared_dataset_root.exists():
        shutil.rmtree(prepared_dataset_root)
    for split in ('train', 'val', 'test'):
        (prepared_dataset_root / split).mkdir(parents=True, exist_ok=True)

    link_fallback_copy_count = 0
    split_counts = {'train': {}, 'val': {}, 'test': {}}

    def _split_counts(total):
        if total < 3:
            return total, 0, 0
        train_count = max(1, int(total * 0.8))
        val_count = max(1, int(total * 0.1))
        if train_count + val_count >= total:
            val_count = 1
            train_count = max(1, total - 2)
        test_count = total - train_count - val_count
        if test_count < 1:
            test_count = 1
            if train_count > 1:
                train_count -= 1
            elif val_count > 0:
                val_count -= 1
        return train_count, val_count, test_count

    for class_name, image_files in class_to_images.items():
        files = list(image_files)
        random.Random(42 + sum(ord(ch) for ch in class_name)).shuffle(files)

        train_n, val_n, test_n = _split_counts(len(files))
        per_split = {'train': files[:train_n], 'val': files[train_n:train_n + val_n], 'test': files[train_n + val_n:]}

        for split_name, split_files in per_split.items():
            class_split_dir = prepared_dataset_root / split_name / class_name
            class_split_dir.mkdir(parents=True, exist_ok=True)
            split_counts[split_name][class_name] = len(split_files)
            for idx, src in enumerate(split_files):
                dst = class_split_dir / f"{idx:06d}_{src.name}"
                try:
                    os.symlink(src, dst)
                except Exception:
                    shutil.copy2(src, dst)
                    link_fallback_copy_count += 1

    metadata = {
        'source_dataset_path': str(source_dataset_root),
        'prepared_dataset_path': str(prepared_dataset_root),
        'source_layout': 'class_root',
        'split_policy': {'train': 0.8, 'val': 0.1, 'test': 0.1},
        'symlink_fallback_copy_count': link_fallback_copy_count,
        'train': {'num_samples': int(sum(split_counts['train'].values())), 'num_classes': len(class_dirs), 'classes': sorted(split_counts['train'].keys()), 'class_counts': split_counts['train']},
        'val': {'num_samples': int(sum(split_counts['val'].values())), 'num_classes': len(class_dirs), 'classes': sorted(split_counts['val'].keys()), 'class_counts': split_counts['val']},
        'test': {'num_samples': int(sum(split_counts['test'].values())), 'num_classes': len(class_dirs), 'classes': sorted(split_counts['test'].keys()), 'class_counts': split_counts['test']},
        'image_size': 224,
        'normalize_mean': [0.485, 0.456, 0.406],
        'normalize_std': [0.229, 0.224, 0.225],
    }

    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)

    template_config_path = REPO_PATH / 'config' / 'colab.json'
    workspace_config_path = workspace_config_dir / 'colab.json'
    if template_config_path.exists():
        with open(template_config_path, 'r', encoding='utf-8') as f:
            colab_config = json.load(f)
    else:
        colab_config = {}

    colab_config.setdefault('colab', {})
    colab_config.setdefault('data', {})
    colab_config.setdefault('training', {})
    colab_config['colab'].setdefault('training', {})
    colab_config['colab'].setdefault('memory_optimization', {})

    colab_config['colab']['workspace_path'] = str(workspace_root)
    colab_config['colab']['drive_mount_path'] = str(PROJECT_ROOT)
    colab_config['colab']['gpu_type'] = 'auto'
    colab_config['colab']['resume_from_checkpoint'] = bool(TRAINING_CONFIG.get('resume_from_checkpoint', True))
    colab_config['colab']['training'].setdefault('gradient_accumulation_steps', 2)
    colab_config['colab']['training'].setdefault('use_amp', True)
    colab_config['colab']['training'].setdefault('pin_memory', True)
    colab_config['colab']['training'].setdefault('num_workers', 2)
    colab_config['colab']['training'].setdefault('prefetch_factor', 2)
    colab_config['colab']['training'].setdefault('early_stopping_patience', 10)
    colab_config['colab']['memory_optimization']['mixed_precision'] = bool(TRAINING_CONFIG.get('mixed_precision', True))

    colab_config['data']['image_size'] = int(colab_config['data'].get('image_size', 224))
    colab_config['data']['normalize_mean'] = metadata['normalize_mean']
    colab_config['data']['normalize_std'] = metadata['normalize_std']
    colab_config['data']['prepared_dataset_path'] = str(prepared_dataset_root)

    phase_defaults = {
        'phase1': {'model_name': 'facebook/dinov3-giant', 'lora_r': 32, 'lora_alpha': 32, 'lora_dropout': 0.1, 'loraplus_lr_ratio': 16},
        'phase2': {'model_name': 'stabilityai/stable-diffusion-2-1', 'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0.1},
        'phase3': {'model_name': 'facebook/dinov3-giant', 'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1, 'checkpoint_interval': 5, 'conec': {'temperature': 0.07, 'prototype_dim': 128, 'num_prototypes': 10, 'contrastive_weight': 0.1, 'orthogonal_weight': 0.01}},
    }

    for phase_name, defaults in phase_defaults.items():
        phase_cfg = colab_config['training'].setdefault(phase_name, {})
        for k, v in defaults.items():
            phase_cfg.setdefault(k, v)
        phase_cfg['batch_size'] = int(TRAINING_CONFIG['batch_size'])
        phase_cfg['learning_rate'] = float(TRAINING_CONFIG['learning_rate'])
        phase_cfg['num_epochs'] = int(TRAINING_CONFIG['num_epochs'])
        phase_cfg['save_best_only'] = bool(TRAINING_CONFIG.get('save_best_only', True))

    class_count = len(class_dirs)
    colab_config['training']['phase1']['num_classes'] = class_count
    colab_config['training']['phase3']['num_classes'] = class_count

    with open(workspace_config_path, 'w', encoding='utf-8') as f:
        json.dump(colab_config, f, indent=2)
    print(f"Workspace config written: {workspace_config_path}")

    workspace_taxonomy = workspace_config_dir / 'plant_taxonomy.json'
    repo_taxonomy = REPO_PATH / 'config' / 'plant_taxonomy.json'
    if repo_taxonomy.exists() and not workspace_taxonomy.exists():
        shutil.copy2(repo_taxonomy, workspace_taxonomy)

    TRAINING_CONFIG['prepared_dataset_path'] = str(prepared_dataset_root)
    TRAINING_CONFIG['dataset_path'] = str(prepared_dataset_root)
    print(f"Prepared split dataset root: {prepared_dataset_root}")
    print(f"Dataset metadata written: {metadata_path}")
    print(f"Symlink fallback copies: {link_fallback_copy_count}")

    checkpoint_manager.save_checkpoint('data_prep', {'source_dataset_path': str(source_dataset_root), 'prepared_dataset_path': str(prepared_dataset_root), 'num_classes': len(class_dirs), 'timestamp': datetime.now().isoformat()})
    print('Data preparation phase complete')

data_elapsed = (datetime.now() - data_start_time).total_seconds() / 60
print(f"Data prep completed in {data_elapsed:.1f} minutes")


---
## 2️⃣ DATA PREPARATION

### Step 2.1: Create Colab-Optimized Dataset

In [ ]:
print()
print('DATA PREPARATION PHASE')
print('=' * 60)
print('Validating prepared dataset splits for phase notebooks...')

data_start_time = datetime.now()

prepared_dataset_root = WORKSPACE_ROOT / 'data' / 'dataset_splits'
required_splits = ('train', 'val', 'test')
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def _has_images(split_path: Path) -> bool:
    return any(p.is_file() and p.suffix.lower() in image_exts for p in split_path.rglob('*'))

missing_splits = [s for s in required_splits if not (prepared_dataset_root / s).exists()]
if missing_splits:
    raise RuntimeError(f"Prepared split dataset missing required folders: {missing_splits}. Run Step 1.6 to generate splits from your Drive dataset.")

empty_splits = [s for s in required_splits if not _has_images(prepared_dataset_root / s)]
if empty_splits:
    raise RuntimeError(f"Prepared split dataset has no images in: {empty_splits}. Run Step 1.6 to regenerate splits from your Drive dataset.")

TRAINING_CONFIG['prepared_dataset_path'] = str(prepared_dataset_root)
TRAINING_CONFIG['dataset_path'] = str(prepared_dataset_root)

print(f"Using prepared dataset splits at: {prepared_dataset_root}")
print('Skipping external dataset download (Drive dataset only).')

data_elapsed = (datetime.now() - data_start_time).total_seconds() / 60
print(f"Data prep completed in {data_elapsed:.1f} minutes")


---
## 3️⃣ PHASE 1: DoRA TRAINING

In [ ]:
print()
print('PHASE 1: DoRA TRAINING')
print('=' * 60)

stage_name = 'phase1'
PIPELINE_STATUS['stages'][stage_name] = 'pending'
phase1_start_time = datetime.now()

if not TRAINING_CONFIG['phases']['phase1']:
    PIPELINE_STATUS['stages'][stage_name] = 'skipped'
    print('Phase 1 skipped (disabled in configuration)')
elif checkpoint_manager.has_checkpoint('phase1'):
    cp = checkpoint_manager.get_checkpoint('phase1')
    PIPELINE_STATUS['stages'][stage_name] = 'checkpoint-skip'
    print(f"Phase 1 already completed at {cp['timestamp']}")
    print('Skipping Phase 1 (checkpoint found)')
else:
    try:
        phase1_nb = REPO_PATH / 'colab_notebooks' / '2_phase1_training.ipynb'
        if phase1_nb.exists():
            import nbformat
            from nbconvert.preprocessors import ExecutePreprocessor
            nb = nbformat.read(str(phase1_nb), as_version=4)
            ep = ExecutePreprocessor(timeout=3600, kernel_name='python3')
            nb, out = ep.preprocess(nb, {'metadata': {'path': str(REPO_PATH / 'colab_notebooks')}})
            PIPELINE_STATUS['stages'][stage_name] = 'completed'
            checkpoint_manager.save_checkpoint('phase1', {'status': 'Completed', 'epochs': TRAINING_CONFIG['num_epochs'], 'crops': list(TRAINING_CONFIG['crops']), 'timestamp': datetime.now().isoformat()})
            print('Phase 1 training completed')
        else:
            msg = f"Phase 1 notebook not found at {phase1_nb}"
            PIPELINE_STATUS['stages'][stage_name] = 'failed'
            PIPELINE_STATUS['errors'].append(msg)
            print(f"ERROR: {msg}")
    except Exception as e:
        msg = f"Phase 1 training error: {e}"
        PIPELINE_STATUS['stages'][stage_name] = 'failed'
        PIPELINE_STATUS['errors'].append(msg)
        print(f"ERROR: {msg}")
        import traceback; traceback.print_exc()

phase1_elapsed = (datetime.now() - phase1_start_time).total_seconds() / 60
print(f"Phase 1 elapsed: {phase1_elapsed:.1f} minutes")
print(f"Phase 1 status: {PIPELINE_STATUS['stages'][stage_name]}")


---
## 4️⃣ PHASE 2: SD-LoRA TRAINING

In [ ]:
print()
print('PHASE 2: SD-LoRA TRAINING')
print('=' * 60)

stage_name = 'phase2'
PIPELINE_STATUS['stages'][stage_name] = 'pending'
phase2_start_time = datetime.now()

if not TRAINING_CONFIG['phases']['phase2']:
    PIPELINE_STATUS['stages'][stage_name] = 'skipped'
    print('Phase 2 skipped (disabled in configuration)')
elif checkpoint_manager.has_checkpoint('phase2'):
    cp = checkpoint_manager.get_checkpoint('phase2')
    PIPELINE_STATUS['stages'][stage_name] = 'checkpoint-skip'
    print(f"Phase 2 already completed at {cp['timestamp']}")
    print('Skipping Phase 2 (checkpoint found)')
else:
    try:
        phase2_nb = REPO_PATH / 'colab_notebooks' / '3_phase2_training.ipynb'
        if phase2_nb.exists():
            import nbformat
            from nbconvert.preprocessors import ExecutePreprocessor
            nb = nbformat.read(str(phase2_nb), as_version=4)
            ep = ExecutePreprocessor(timeout=3600, kernel_name='python3')
            nb, out = ep.preprocess(nb, {'metadata': {'path': str(REPO_PATH / 'colab_notebooks')}})
            PIPELINE_STATUS['stages'][stage_name] = 'completed'
            checkpoint_manager.save_checkpoint('phase2', {'status': 'Completed', 'epochs': TRAINING_CONFIG['num_epochs'], 'crops': list(TRAINING_CONFIG['crops']), 'timestamp': datetime.now().isoformat()})
            print('Phase 2 training completed')
        else:
            msg = f"Phase 2 notebook not found at {phase2_nb}"
            PIPELINE_STATUS['stages'][stage_name] = 'failed'
            PIPELINE_STATUS['errors'].append(msg)
            print(f"ERROR: {msg}")
    except Exception as e:
        msg = f"Phase 2 training error: {e}"
        PIPELINE_STATUS['stages'][stage_name] = 'failed'
        PIPELINE_STATUS['errors'].append(msg)
        print(f"ERROR: {msg}")
        import traceback; traceback.print_exc()

phase2_elapsed = (datetime.now() - phase2_start_time).total_seconds() / 60
print(f"Phase 2 elapsed: {phase2_elapsed:.1f} minutes")
print(f"Phase 2 status: {PIPELINE_STATUS['stages'][stage_name]}")


---
## 5️⃣ PHASE 3: CoNeC-LoRA TRAINING

In [ ]:
print()
print('PHASE 3: CoNeC-LoRA TRAINING')
print('=' * 60)

stage_name = 'phase3'
PIPELINE_STATUS['stages'][stage_name] = 'pending'
phase3_start_time = datetime.now()

if not TRAINING_CONFIG['phases']['phase3']:
    PIPELINE_STATUS['stages'][stage_name] = 'skipped'
    print('Phase 3 skipped (disabled in configuration)')
elif checkpoint_manager.has_checkpoint('phase3'):
    cp = checkpoint_manager.get_checkpoint('phase3')
    PIPELINE_STATUS['stages'][stage_name] = 'checkpoint-skip'
    print(f"Phase 3 already completed at {cp['timestamp']}")
    print('Skipping Phase 3 (checkpoint found)')
else:
    try:
        phase3_nb = REPO_PATH / 'colab_notebooks' / '4_phase3_training.ipynb'
        if phase3_nb.exists():
            import nbformat
            from nbconvert.preprocessors import ExecutePreprocessor
            nb = nbformat.read(str(phase3_nb), as_version=4)
            ep = ExecutePreprocessor(timeout=3600, kernel_name='python3')
            nb, out = ep.preprocess(nb, {'metadata': {'path': str(REPO_PATH / 'colab_notebooks')}})
            PIPELINE_STATUS['stages'][stage_name] = 'completed'
            checkpoint_manager.save_checkpoint('phase3', {'status': 'Completed', 'epochs': TRAINING_CONFIG['num_epochs'], 'crops': list(TRAINING_CONFIG['crops']), 'timestamp': datetime.now().isoformat()})
            print('Phase 3 training completed')
        else:
            msg = f"Phase 3 notebook not found at {phase3_nb}"
            PIPELINE_STATUS['stages'][stage_name] = 'failed'
            PIPELINE_STATUS['errors'].append(msg)
            print(f"ERROR: {msg}")
    except Exception as e:
        msg = f"Phase 3 training error: {e}"
        PIPELINE_STATUS['stages'][stage_name] = 'failed'
        PIPELINE_STATUS['errors'].append(msg)
        print(f"ERROR: {msg}")
        import traceback; traceback.print_exc()

phase3_elapsed = (datetime.now() - phase3_start_time).total_seconds() / 60
print(f"Phase 3 elapsed: {phase3_elapsed:.1f} minutes")
print(f"Phase 3 status: {PIPELINE_STATUS['stages'][stage_name]}")


---
## 6️⃣ VALIDATION & TESTING

In [ ]:
print()
print('VALIDATION & TESTING PHASE')
print('=' * 60)

stage_name = 'validation'
PIPELINE_STATUS['stages'][stage_name] = 'pending'
validation_start_time = datetime.now()

if not TRAINING_CONFIG['validate']:
    PIPELINE_STATUS['stages'][stage_name] = 'skipped'
    print('Validation skipped (disabled in configuration)')
elif checkpoint_manager.has_checkpoint('validation'):
    cp = checkpoint_manager.get_checkpoint('validation')
    PIPELINE_STATUS['stages'][stage_name] = 'checkpoint-skip'
    print(f"Validation already completed at {cp['timestamp']}")
    print('Skipping validation (checkpoint found)')
else:
    try:
        validation_nb = REPO_PATH / 'colab_notebooks' / '5_testing_validation.ipynb'
        if validation_nb.exists():
            import nbformat
            from nbconvert.preprocessors import ExecutePreprocessor
            nb = nbformat.read(str(validation_nb), as_version=4)
            ep = ExecutePreprocessor(timeout=1800, kernel_name='python3')
            nb, out = ep.preprocess(nb, {'metadata': {'path': str(REPO_PATH / 'colab_notebooks')}})
            PIPELINE_STATUS['stages'][stage_name] = 'completed'
            checkpoint_manager.save_checkpoint('validation', {'status': 'Completed', 'crops': list(TRAINING_CONFIG['crops']), 'timestamp': datetime.now().isoformat()})
            print('Validation completed')
        else:
            msg = f"Validation notebook not found at {validation_nb}"
            PIPELINE_STATUS['stages'][stage_name] = 'failed'
            PIPELINE_STATUS['warnings'].append(msg)
            print(f"WARNING: {msg}")
    except Exception as e:
        msg = f"Validation warning: {e}"
        PIPELINE_STATUS['stages'][stage_name] = 'failed'
        PIPELINE_STATUS['warnings'].append(msg)
        print(f"WARNING: {msg}")
        print('Continuing with performance monitoring...')

validation_elapsed = (datetime.now() - validation_start_time).total_seconds() / 60
print(f"Validation elapsed: {validation_elapsed:.1f} minutes")
print(f"Validation status: {PIPELINE_STATUS['stages'][stage_name]}")


---
## 7️⃣ PERFORMANCE MONITORING & REPORTS

In [ ]:
print()
print('PERFORMANCE MONITORING & REPORTS')
print('=' * 60)

stage_name = 'monitoring'
PIPELINE_STATUS['stages'][stage_name] = 'pending'
monitoring_start_time = datetime.now()

if checkpoint_manager.has_checkpoint('monitoring'):
    cp = checkpoint_manager.get_checkpoint('monitoring')
    PIPELINE_STATUS['stages'][stage_name] = 'checkpoint-skip'
    print(f"Performance monitoring already completed at {cp['timestamp']}")
    print('Skipping monitoring (checkpoint found)')
else:
    try:
        monitoring_nb = REPO_PATH / 'colab_notebooks' / '6_performance_monitoring.ipynb'
        if monitoring_nb.exists():
            import nbformat
            from nbconvert.preprocessors import ExecutePreprocessor
            nb = nbformat.read(str(monitoring_nb), as_version=4)
            ep = ExecutePreprocessor(timeout=1800, kernel_name='python3')
            nb, out = ep.preprocess(nb, {'metadata': {'path': str(REPO_PATH / 'colab_notebooks')}})
            PIPELINE_STATUS['stages'][stage_name] = 'completed'
            checkpoint_manager.save_checkpoint('monitoring', {'status': 'Completed', 'output_directory': TRAINING_CONFIG['output_directory'], 'timestamp': datetime.now().isoformat()})
            print('Performance monitoring completed')
        else:
            msg = f"Monitoring notebook not found at {monitoring_nb}"
            PIPELINE_STATUS['stages'][stage_name] = 'failed'
            PIPELINE_STATUS['warnings'].append(msg)
            print(f"WARNING: {msg}")
    except Exception as e:
        msg = f"Monitoring warning: {e}"
        PIPELINE_STATUS['stages'][stage_name] = 'failed'
        PIPELINE_STATUS['warnings'].append(msg)
        print(f"WARNING: {msg}")

monitoring_elapsed = (datetime.now() - monitoring_start_time).total_seconds() / 60
print(f"Monitoring elapsed: {monitoring_elapsed:.1f} minutes")
print(f"Monitoring status: {PIPELINE_STATUS['stages'][stage_name]}")


---
## 🎉 FINAL SUMMARY & RESULTS

In [ ]:
total_elapsed = (datetime.now() - PIPELINE_START_TIME).total_seconds() / 60

if PIPELINE_STATUS['errors']:
    print()
    print('=' * 60)
    print('TRAINING PIPELINE COMPLETED WITH ERRORS')
    print('=' * 60)
else:
    print()
    print('=' * 60)
    print('TRAINING PIPELINE COMPLETED SUCCESSFULLY')
    print('=' * 60)

print()
print('CONFIGURATION USED:')
print(f"  Crops: {', '.join(TRAINING_CONFIG['crops'])}")
if TRAINING_CONFIG['crops'] and TRAINING_CONFIG['crops'][-1] == 'custom' and TRAINING_CONFIG['custom_crop']:
    print(f"  Including custom crop: {TRAINING_CONFIG['custom_crop']}")
print(f"  Source Dataset Path: {TRAINING_CONFIG.get('source_dataset_path')}")
print(f"  Prepared Dataset Path: {TRAINING_CONFIG.get('prepared_dataset_path')}")
print(f"  Batch Size: {TRAINING_CONFIG['batch_size']}")
print(f"  Learning Rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  Epochs per Phase: {TRAINING_CONFIG['num_epochs']}")
print(f"  Device: {TRAINING_CONFIG['device']}")
print(f"  Mixed Precision: {'Enabled' if TRAINING_CONFIG['mixed_precision'] else 'Disabled'}")

print()
print('CHECKPOINT STATUS:')
checkpoint_manager.display_checkpoint_status()

print()
print('STAGE STATUS:')
for stage, status in PIPELINE_STATUS['stages'].items():
    print(f"  - {stage}: {status}")

if PIPELINE_STATUS['warnings']:
    print()
    print('WARNINGS:')
    for item in PIPELINE_STATUS['warnings']:
        print(f"  - {item}")

if PIPELINE_STATUS['errors']:
    print()
    print('ERRORS:')
    for item in PIPELINE_STATUS['errors']:
        print(f"  - {item}")

with open(TRAINING_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump({'created_at': PIPELINE_START_TIME.isoformat(), 'finished_at': datetime.now().isoformat(), 'config': TRAINING_CONFIG, 'pipeline_status': PIPELINE_STATUS, 'total_runtime_minutes': total_elapsed}, f, indent=2)

print()
print('EXECUTION SUMMARY:')
print(f"  Total Runtime: ~{total_elapsed:.1f} minutes")
print(f"  Project Root: {PROJECT_ROOT}")
print(f"  Output Directory: {TRAINING_CONFIG['output_directory']}")
print(f"  Models Directory: {MODELS_DIR}")
print(f"  Logs Directory: {LOGS_DIR}")
print(f"  Checkpoints: {TRAINING_CONFIG['checkpoint_directory']}")
print(f"  Configuration Record: {TRAINING_CONFIG_PATH}")
print(f"  Checkpoint Log: {TRAINING_CONFIG['checkpoint_directory']}/checkpoint_log.json")

print()
print('=' * 60)
print('Pipeline run finished')
print('=' * 60)
